## MOVIE  RECOMMENDATION SYSTEM 

Use of each dataset:
1. movies.csv - Metadata for Content-Based Filtering. 
2. rating.csv - Core data for Collaborative Filtering. 
3. tags.csv - User Tags.
4. links.csv - Links to fetch posters/synopses via the TMDB API later.

In [57]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [58]:
movies = pd.read_csv(r'C:\Abhishree\Projects_GitHubLinked\Movie Recommender\data\ml-latest-small\movies.csv')
ratings = pd.read_csv(r'C:\Abhishree\Projects_GitHubLinked\Movie Recommender\data\ml-latest-small\ratings.csv')

In [59]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [60]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [61]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='str')

In [62]:
ratings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')

In [63]:
movies['genres'].value_counts()

genres
Drama                                     1053
Comedy                                     946
Comedy|Drama                               435
Comedy|Romance                             363
Drama|Romance                              349
                                          ... 
Adventure|Mystery|Sci-Fi|Thriller            1
Action|Comedy|Crime|Horror                   1
Action|Adventure|Children|Sci-Fi             1
Action|Adventure|Comedy|Fantasy|Sci-Fi       1
Action|Animation|Comedy|Fantasy              1
Name: count, Length: 951, dtype: int64

## one hot encode of genres. 

In [64]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [65]:
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

In [66]:
movies.head()

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II


## CONTENT BASED RECOMMENDATION

1. convert words/features to numbers(vectors) and then compare the angle between them.
STEP A: TF-IDF
* It converts text into numerical vectors based on how important a word is to a document relative to all documents.
* TF-IDF(t, d, D) = TF(t, d)*IDF(t, D)
* TF (Term Frequency): How often a word appears in a single item description.
* IDF (Inverse Document Frequency): Penalizes common words (like "the", "movie", "a") that appear across every item.

In [74]:
movies.head()

,movieId,title,genres,clean_title,clean_genre
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy


>Clean the 'genres' column and remove this character -> '|'.

In [75]:
movies['clean_genre'] = movies['genres'].str.replace("|"," ", regex=False)
movies.head()

,movieId,title,genres,clean_title,clean_genre
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy


### VECTORIZATION
Turn text into numbers. We use TF-IDF

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer 

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['clean_genre'])
tfidf_matrix.shape

(9742, 24)

### COSINE SIMILARITY
* Calculate the similarity between all movies. 
* If two movies have identical genres, their similarity score is 1.0. 
* If they share no genres at all, their score is 0.0

In [77]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix,tfidf_matrix)
cosine_sim.shape

(9742, 9742)

### RECOMMENDATION SYSTEM

In [78]:
movies.head()

,movieId,title,genres,clean_title,clean_genre
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy


In [85]:
def recomender(title,top_n =5): 
    matches = movies[movies['clean_title'].str.contains(title,case=False,regex=False)]
    if matches.empty:
        return 'Movie not found'
    movie_idx = matches.index[0]     
    movie_score = list(enumerate(cosine_sim[movie_idx]))

    sorted_score = sorted(movie_score, key= lambda x: x[1], reverse=True)

    top_movie = [i[0] for i in sorted_score[1:top_n+1]] 

    return movies['clean_title'].iloc[top_movie]

In [86]:
movies['clean_title'].head(10)

0                      Toy Story
1                        Jumanji
2               Grumpier Old Men
3              Waiting to Exhale
4    Father of the Bride Part II
5                           Heat
6                        Sabrina
7                   Tom and Huck
8                   Sudden Death
9                      GoldenEye
Name: clean_title, dtype: str

In [89]:
print(recomender('inception'))

6797          Watchmen
7625           Super 8
8358           RoboCop
167       Strange Days
6151    V for Vendetta
Name: clean_title, dtype: str
